# Few-Shot Segmentation: Euclidean vs Pontryagin — UAV Sugarcane

```
0. GPU check
1. Mount Google Drive
2. Configuration  ← EDIT paths here
3. Clone repo + install deps
4. Dataset
5. Smoke tests
6. HPO  (optional, ~30-60 min por modelo en T4)
7. Full training  (loads best_params.json automatically)
8. Analysis & results  (saved to Drive)
9. Download
```

> **Only cell 2 needs to be edited before running.**

## 0 · GPU check

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: no GPU — Runtime → Change runtime type → GPU')

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Configuration  ← EDIT THIS CELL

Set your Drive paths here.  
Everything else runs automatically.

In [ ]:
import os

# ── Repository ────────────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/craljimenez/rr-pontryagin-embedded.git'
REPO_DIR = '/content/rr-pontryagin-embedded'

# ── Google Drive paths ────────────────────────────────────────────────────────
# Dataset root — must contain train/, valid/, test/ (YOLO polygon format)
DATA_ROOT          = '/content/drive/MyDrive/UAV_segmantation'
# All checkpoints, metrics and figures go here (persists across sessions)
DRIVE_RESULTS_ROOT = '/content/drive/MyDrive/fss_results'

# ── Experiment ────────────────────────────────────────────────────────────────
K_SHOT = 1   # support images per episode (1 or 5)

# ─────────────────────────────────────────────────────────────────────────────
os.makedirs(DRIVE_RESULTS_ROOT, exist_ok=True)
print('Config OK.')
print(f'  DATA_ROOT          : {DATA_ROOT}')
print(f'  DRIVE_RESULTS_ROOT : {DRIVE_RESULTS_ROOT}')
print(f'  K_SHOT             : {K_SHOT}')

## 3 · Clone repo + install dependencies

In [ ]:
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --rebase
print(f'Repo at: {REPO_DIR}')

In [ ]:
!pip install -e {REPO_DIR} -q
!pip install scikit-optimize -q
print('Dependencies installed.')

In [ ]:
import sys, os
from pathlib import Path

for p in [f'{REPO_DIR}/experiments', f'{REPO_DIR}/src']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(f'{REPO_DIR}/experiments')

from configs.fss_sugarcane import AVAILABLE_MODELS
print('Imports OK. Available models:', AVAILABLE_MODELS)

## 4 · Dataset

Verifica que `DATA_ROOT` contenga las splits `train/`, `valid/`, `test/` en formato YOLO polygon.  
Clase objetivo: **5 — Sugarcane**.

In [ ]:
from pathlib import Path
from prfe.data.fss_dataset import EpisodicUAVDataset

DATA = Path(DATA_ROOT)
ok   = True
print('Split summary:')
for split in ('train', 'valid', 'test'):
    imgs = list((DATA / split / 'images').glob('*.jpg'))
    lbls = list((DATA / split / 'labels').glob('*.txt'))
    ds   = EpisodicUAVDataset(DATA, split, k_shot=K_SHOT, n_episodes=4, img_size=128)
    status = '✓' if imgs else '✗'
    if not imgs: ok = False
    print(f'  {status} {split:6s}: {len(imgs):3d} imgs  {len(lbls):3d} labels  '
          f'{len(ds.samples):3d} eligible (sugarcane)')

if not ok:
    raise RuntimeError('Dataset incompleto — revisa DATA_ROOT')
print('\nDataset OK.')

## 5 · Smoke tests  *(~2 min en CPU)*

Verifica dataset, forward pass, losses y gradientes con tensores 64×64.  
Deben completarse antes de lanzar HPO o entrenamiento.

In [ ]:
import torch
from torch.utils.data import DataLoader
from prfe.data.fss_dataset import EpisodicUAVDataset
from prfe.models.fss import EuclideanFewShotSeg, PontryaginFewShotSeg
from prfe.losses.fss import EuclideanFSSLoss, PontryaginFSSLoss

# ── Test 1: dataset shapes ────────────────────────────────────────────────────
ds    = EpisodicUAVDataset(DATA, 'train', k_shot=K_SHOT, n_episodes=8, img_size=128)
batch = next(iter(DataLoader(ds, batch_size=4, num_workers=0)))
assert batch['support_imgs'].shape  == (4, K_SHOT, 3, 128, 128)
assert batch['query_mask'].shape    == (4, 128, 128)
print(f'✓ Dataset  support={tuple(batch["support_imgs"].shape)}  '
      f'query={tuple(batch["query_img"].shape)}')

# ── Test 2: forward pass ──────────────────────────────────────────────────────
B, K, H, W = 2, K_SHOT, 64, 64
si = torch.randn(B, K, 3, H, W)
sm = torch.randint(0, 2, (B, K, H, W)).float()
qi = torch.randn(B, 3, H, W)
qm = torch.randint(0, 2, (B, H, W)).float()

m_euc  = EuclideanFewShotSeg(base_ch=32, depth=3)
m_pont = PontryaginFewShotSeg(base_ch=32, depth=3, n_rff=16, n_srf=4, kappa=4)
assert m_euc(si, sm, qi).shape  == (B, H, W)
assert m_pont(si, sm, qi).shape == (B, H, W)
print(f'✓ Forward pass  euc={tuple(m_euc(si,sm,qi).shape)}  '
      f'pont={tuple(m_pont(si,sm,qi).shape)}')

# ── Test 3: losses & gradients ───────────────────────────────────────────────
l_euc  = EuclideanFSSLoss(bce_weight=0.5, pos_weight=5.0)
l_pont = PontryaginFSSLoss(J=m_pont.J, bce_weight=0.5, pos_weight=5.0,
                           lambda_cone=0.1, lambda_orth=0.05)
loss_e = m_euc.compute_loss(si, sm, qi, qm, l_euc);  loss_e.backward()
loss_p = m_pont.compute_loss(si, sm, qi, qm, l_pont); loss_p.backward()
assert torch.isfinite(loss_e) and torch.isfinite(loss_p)
print(f'✓ Losses  euc={loss_e.item():.4f}  pont={loss_p.item():.4f}')

print('\n✓ All smoke tests passed.')

## 6 · HPO  *(opcional — ~30-60 min por modelo en T4)*

Optimización bayesiana con `scikit-optimize` sobre val IoU.  
Salta automáticamente si `best_params.json` ya existe en Drive.  
Puedes saltar esta sección entera para entrenar con los hiperparámetros del config.

In [ ]:
import json
from pathlib import Path

for mt in AVAILABLE_MODELS:
    p = Path(DRIVE_RESULTS_ROOT) / f'{mt}_{K_SHOT}shot' / 'hpo' / 'best_params.json'
    if p.exists():
        d     = json.loads(p.read_text())
        score = d.get('_best_val_iou', float('nan'))
        print(f'  {mt:<12}  HPO done  —  best val_iou = {score:.4f}')
    else:
        print(f'  {mt:<12}  no HPO results yet')

### 6a · Euclidean

In [ ]:
from pathlib import Path
if not (Path(DRIVE_RESULTS_ROOT) / f'euclidean_{K_SHOT}shot' / 'hpo' / 'best_params.json').exists():
    !python {REPO_DIR}/experiments/tune_fss_sugarcane.py \
        --model        euclidean \
        --k-shot       {K_SHOT} \
        --n-calls      20 \
        --n-random     8 \
        --trial-epochs 8 \
        --data-root    "{DATA_ROOT}" \
        --results-dir  "{DRIVE_RESULTS_ROOT}"
else:
    print('euclidean HPO already done — skipping.')

### 6b · Pontryagin

In [ ]:
from pathlib import Path
if not (Path(DRIVE_RESULTS_ROOT) / f'pontryagin_{K_SHOT}shot' / 'hpo' / 'best_params.json').exists():
    !python {REPO_DIR}/experiments/tune_fss_sugarcane.py \
        --model        pontryagin \
        --k-shot       {K_SHOT} \
        --n-calls      30 \
        --n-random     10 \
        --trial-epochs 8 \
        --data-root    "{DATA_ROOT}" \
        --results-dir  "{DRIVE_RESULTS_ROOT}"
else:
    print('pontryagin HPO already done — skipping.')

## 7 · Entrenamiento completo

Carga `best_params.json` automáticamente si el HPO fue ejecutado.  
En caso contrario usa los valores por defecto del config.  
~20-30 min por modelo en T4 con `K_SHOT=1`.

In [ ]:
!python {REPO_DIR}/experiments/run_fss_sugarcane.py \
    --model       euclidean \
    --k-shot      {K_SHOT} \
    --data-root   "{DATA_ROOT}" \
    --results-dir "{DRIVE_RESULTS_ROOT}"

In [ ]:
!python {REPO_DIR}/experiments/run_fss_sugarcane.py \
    --model       pontryagin \
    --k-shot      {K_SHOT} \
    --data-root   "{DATA_ROOT}" \
    --results-dir "{DRIVE_RESULTS_ROOT}"

## 8 · Análisis y resultados

### 8a · Tabla de métricas

In [ ]:
import json
import pandas as pd
from pathlib import Path

rows = []
for mt in AVAILABLE_MODELS:
    p = Path(DRIVE_RESULTS_ROOT) / f'{mt}_{K_SHOT}shot' / 'metrics_test.json'
    if p.exists():
        d = json.loads(p.read_text())
        rows.append({
            'model':       mt,
            'k_shot':      d['k_shot'],
            'best_val_iou': round(d['best_val_iou'],   4),
            'test_iou':    round(d['test_iou'],        4),
            'test_dice':   round(d['test_dice'],       4),
            'precision':   round(d['test_precision'],  4),
            'recall':      round(d['test_recall'],     4),
        })
    else:
        rows.append({'model': mt, 'nota': 'sin entrenar / no encontrado'})

print('=== Test-set metrics ===')
print(pd.DataFrame(rows).set_index('model').to_string())

### 8b · Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

COLORS = {'euclidean': '#2196F3', 'pontryagin': '#FF5722'}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

for mt, color in COLORS.items():
    csv_path = Path(DRIVE_RESULTS_ROOT) / f'{mt}_{K_SHOT}shot' / 'metrics.csv'
    if not csv_path.exists():
        print(f'  {mt}: metrics.csv not found — skipping'); continue
    df = pd.read_csv(csv_path)
    ax1.plot(df['epoch'], df['train_loss'], label=mt.capitalize(), color=color)
    ax2.plot(df['epoch'], df['val_iou'],    label=mt.capitalize(), color=color)

for ax, title, ylabel in [
    (ax1, 'Training Loss', 'Loss'),
    (ax2, f'Validation IoU  ({K_SHOT}-shot)', 'IoU'),
]:
    ax.set_title(title, fontsize=12); ax.set_xlabel('Época')
    ax.set_ylabel(ylabel); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
out = Path(DRIVE_RESULTS_ROOT) / f'curves_{K_SHOT}shot.png'
plt.savefig(out, dpi=120, bbox_inches='tight')
plt.show()
print(f'Guardado: {out}')

### 8c · Evaluación estadística sobre test

Corre 200 episodios de test, acumula métricas por episodio y aplica un Welch t-test.  
Las figuras se guardan en Drive.

In [ ]:
import torch, numpy as np, json
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from prfe.data.fss_dataset import EpisodicUAVDataset
from run_fss_sugarcane import build_model
import configs.fss_sugarcane as cfg

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(DEVICE)
N_TEST = 200
THRESH = 0.5

# ── carga modelos ─────────────────────────────────────────────────────────────
models = {}
for mt in AVAILABLE_MODELS:
    ckpt = Path(DRIVE_RESULTS_ROOT) / f'{mt}_{K_SHOT}shot' / 'best_model.pth'
    if not ckpt.exists():
        print(f'✗  {mt}: checkpoint not found — skipping'); continue
    # Reconstruct the exact architecture used during training.
    # HPO may have changed rff_multiplier / srf_multiplier which alter tensor
    # dimensions — loading with wrong dims raises a state_dict size mismatch.
    hpo_json = Path(DRIVE_RESULTS_ROOT) / f'{mt}_{K_SHOT}shot' / 'hpo' / 'best_params.json'
    best_p = None
    if hpo_json.exists():
        best_p = {k: v for k, v in json.loads(hpo_json.read_text()).items()
                  if not k.startswith('_')}
    m, _ = build_model(mt, K_SHOT, device, params=best_p)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval(); models[mt] = m
    print(f'✓  {mt}: loaded ({ckpt.stat().st_size/1024**2:.1f} MB)')

assert models, 'No models loaded — train first (section 7)'

# ── evaluación por episodio ───────────────────────────────────────────────────
test_ds     = EpisodicUAVDataset(DATA, 'test', k_shot=K_SHOT,
                                  n_episodes=N_TEST, img_size=cfg.IMG_SIZE, seed=42)
test_loader = DataLoader(test_ds, batch_size=4, num_workers=2, pin_memory=True)

def _metrics(pred, gt):
    pred = (pred > THRESH).float()
    tp   = (pred * gt).sum(dim=(1,2))
    fp   = (pred * (1-gt)).sum(dim=(1,2))
    fn   = ((1-pred) * gt).sum(dim=(1,2))
    eps  = 1e-6
    return ((tp/(tp+fp+fn+eps)).cpu().numpy(),
            (2*tp/(2*tp+fp+fn+eps)).cpu().numpy(),
            (tp/(tp+fp+eps)).cpu().numpy(),
            (tp/(tp+fn+eps)).cpu().numpy())

all_m = {mt: {'iou':[], 'dice':[], 'precision':[], 'recall':[]} for mt in models}

for batch in tqdm(test_loader, desc='Test episodes'):
    si = batch['support_imgs'].to(device)
    sm = batch['support_masks'].to(device)
    qi = batch['query_img'].to(device)
    gt = batch['query_mask'].to(device)
    for mt, model in models.items():
        with torch.no_grad():
            logits = model(si, sm, qi)
        for key, arr in zip(['iou','dice','precision','recall'], _metrics(logits, gt)):
            all_m[mt][key].extend(arr)

for mt in all_m:
    for k in all_m[mt]:
        all_m[mt][k] = np.array(all_m[mt][k])

print(f'\nEvaluación completada  ({N_TEST} episodios, umbral={THRESH})')
for mt, m in all_m.items():
    print(f'\n  {mt.capitalize()}')
    for metric, vals in m.items():
        print(f'    {metric:10s}: {vals.mean():.4f} ± {vals.std():.4f}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats as scipy_stats
import pandas as pd

METRICS = ['iou', 'dice', 'precision', 'recall']
LABELS  = {'iou': 'IoU', 'dice': 'Dice', 'precision': 'Precision', 'recall': 'Recall'}
COLORS  = {'euclidean': '#2196F3', 'pontryagin': '#FF5722'}

# ── tabla estadística ─────────────────────────────────────────────────────────
rows = []
for mt, m in all_m.items():
    row = {'Modelo': mt.capitalize()}
    for met in METRICS:
        row[f'{LABELS[met]} mean'] = f"{m[met].mean():.4f}"
        row[f'{LABELS[met]} std']  = f"{m[met].std():.4f}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

# ── Welch t-test ──────────────────────────────────────────────────────────────
mts = list(all_m.keys())
if len(mts) == 2:
    print('\nWelch t-test:')
    for met in METRICS:
        a, b = all_m[mts[0]][met], all_m[mts[1]][met]
        t, p = scipy_stats.ttest_ind(a, b, equal_var=False)
        sig  = '**p<0.01**' if p < 0.01 else ('*p<0.05*' if p < 0.05 else 'n.s.')
        print(f'  {LABELS[met]:12s}: t={t:+.3f}  p={p:.4f}  {sig}'
              f'  Δ={b.mean()-a.mean():+.4f}  (Pont-Euc)')

# ── figura: histogramas + boxplots ────────────────────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

for col, met in enumerate(METRICS):
    ax_hist = fig.add_subplot(gs[0, col])
    for mt, color in COLORS.items():
        if mt not in all_m: continue
        vals = all_m[mt][met]
        ax_hist.hist(vals, bins=25, alpha=0.55, color=color,
                     label=mt.capitalize(), density=True)
        ax_hist.axvline(vals.mean(), color=color, lw=2, ls='--')
    ax_hist.set_title(LABELS[met], fontsize=11, fontweight='bold')
    ax_hist.set_xlabel('Valor por episodio')
    if col == 0: ax_hist.set_ylabel('Densidad'); ax_hist.legend(fontsize=9)
    ax_hist.grid(alpha=0.25)

    ax_box = fig.add_subplot(gs[1, col])
    bp = ax_box.boxplot([all_m[mt][met] for mt in all_m],
                        patch_artist=True, widths=0.5,
                        medianprops=dict(color='white', lw=2))
    for patch, mt in zip(bp['boxes'], all_m):
        patch.set_facecolor(COLORS.get(mt, 'gray')); patch.set_alpha(0.75)
    ax_box.set_xticklabels([mt.capitalize() for mt in all_m], fontsize=9)
    ax_box.set_title(f'{LABELS[met]} — boxplot', fontsize=10)
    if col == 0: ax_box.set_ylabel(LABELS[met])
    ax_box.grid(axis='y', alpha=0.25)

fig.suptitle(f'Euclidean vs Pontryagin — {N_TEST} episodios test  ({K_SHOT}-shot)',
             fontsize=13, y=1.01)
out_fig = Path(DRIVE_RESULTS_ROOT) / f'test_analysis_{K_SHOT}shot.png'
plt.savefig(out_fig, dpi=130, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_fig}')

# ── JSON detallado ────────────────────────────────────────────────────────────
summary = {}
for mt, m in all_m.items():
    summary[mt] = {
        met: {'mean': float(m[met].mean()), 'std':  float(m[met].std()),
              'min':  float(m[met].min()),  'max':  float(m[met].max()),
              'median': float(np.median(m[met]))}
        for met in METRICS
    }
    summary[mt]['n_episodes'] = int(len(m['iou']))
out_json = Path(DRIVE_RESULTS_ROOT) / f'test_detailed_{K_SHOT}shot.json'
out_json.write_text(json.dumps(summary, indent=2))
print(f'Guardado: {out_json}')

### 8d · Comparación visual

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from prfe.data.fss_dataset import EpisodicUAVDataset
import configs.fss_sugarcane as cfg

N_VIZ  = 6
viz_ds = EpisodicUAVDataset(DATA, 'test', k_shot=K_SHOT,
                             n_episodes=N_VIZ, img_size=cfg.IMG_SIZE, seed=123)

_MEAN  = np.array([0.485, 0.456, 0.406])
_STD   = np.array([0.229, 0.224, 0.225])
denorm = lambda t: np.clip(t.permute(1,2,0).cpu().numpy() * _STD + _MEAN, 0, 1)

n_col      = K_SHOT + 2 + len(models)
col_titles = ([f'Soporte {k+1}' for k in range(K_SHOT)]
              + ['Query', 'GT']
              + [f'Pred\n{mt.capitalize()}' for mt in models])

fig, axes = plt.subplots(N_VIZ, n_col, figsize=(3.0*n_col, 3.0*N_VIZ))
if N_VIZ == 1: axes = axes[None]

for ei, ep in enumerate(viz_ds):
    si = ep['support_imgs'].unsqueeze(0).to(device)
    sm = ep['support_masks'].unsqueeze(0).to(device)
    qi = ep['query_img'].unsqueeze(0).to(device)

    for k in range(K_SHOT):
        axes[ei, k].imshow(denorm(ep['support_imgs'][k]))
        mask_k  = ep['support_masks'][k].numpy()
        overlay = np.zeros((*mask_k.shape, 4))
        overlay[..., 1] = 0.8 * mask_k; overlay[..., 3] = 0.4 * mask_k
        axes[ei, k].imshow(overlay); axes[ei, k].axis('off')

    axes[ei, K_SHOT].imshow(denorm(ep['query_img']));             axes[ei, K_SHOT].axis('off')
    axes[ei, K_SHOT+1].imshow(ep['query_mask'].numpy(),
                               cmap='Greens', vmin=0, vmax=1);    axes[ei, K_SHOT+1].axis('off')

    for ci, (mt, model) in enumerate(models.items()):
        with torch.no_grad():
            pred = model.predict(si, sm, qi).squeeze(0).cpu().numpy()
        gt    = ep['query_mask'].numpy()
        iou_e = (pred*gt).sum() / (pred+gt-pred*gt).sum().clip(min=1e-6)
        ax    = axes[ei, K_SHOT+2+ci]
        ax.imshow(pred, cmap='Greens', vmin=0, vmax=1)
        ax.set_title(f'IoU={iou_e:.3f}', fontsize=8, pad=2)
        ax.axis('off')

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=9, fontweight='bold', pad=6)

plt.suptitle(f'Predicciones {K_SHOT}-shot — Caña de azúcar  (verde = foreground)',
             fontsize=12, y=1.01)
plt.tight_layout()
out = Path(DRIVE_RESULTS_ROOT) / f'qualitative_{K_SHOT}shot.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
plt.show()
print(f'Guardado: {out}')

## 9 · Descargar / explorar resultados

La copia en Drive ya es persistente. Esta celda permite descargar un zip a tu máquina local.

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

zip_path = '/content/fss_sugarcane_results'
shutil.make_archive(zip_path, 'zip', DRIVE_RESULTS_ROOT)
size_mb = Path(zip_path + '.zip').stat().st_size / 1e6
print(f'Archive: {zip_path}.zip  ({size_mb:.1f} MB)')
files.download(zip_path + '.zip')

In [ ]:
# Explorar árbol de archivos en Drive
import os
for root, dirs, fnames in os.walk(DRIVE_RESULTS_ROOT):
    level  = root.replace(DRIVE_RESULTS_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        for f in sorted(fnames):
            size = os.path.getsize(os.path.join(root, f)) / 1024
            print(f'{indent}  {f:<45} {size:7.1f} KB')